## Minimal Meeting Simulation

In [11]:
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

# ======================
# Setup
# ======================

load_dotenv(override=True)

# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.getenv("OPENROUTER_API_KEY")
# )

# MODEL = "openai/gpt-oss-20b:free"

# 1. 將 base_url 指向 Ollama 本地服務位址 (預設為 http://localhost:11434/v1)
# 2. api_key 在本地 Ollama 中雖然不被驗證，但仍需填寫一個非空字串 (例如 "ollama")
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# 3. 將 MODEL 設定為你本地已拉取的模型名稱
MODEL = "ministral-3:8b"


TOPIC = "Should we implement a three-layer abstraction architecture for our multi-agent LLM system?"

# ======================
# Agent Definitions
# ======================

AGENTS = {
    "Architect": {
        "identity": """You are a software architect.
You care about scalability, abstraction, and long-term maintainability.
You prefer clean layered design.
Speak concisely in 1–2 sentences."""
    },
    "Engineer": {
        "identity": """You are a pragmatic software engineer.
You care about simplicity and fast implementation.
You dislike over-engineering.
Speak concisely in 1–2 sentences."""
    },
    "Researcher": {
        "identity": """You are an AI researcher.
You care about model performance and advanced capabilities.
You are interested in memory, reflection, and agent design.
Speak concisely in 1–2 sentences."""
    }
}

# ======================
# Conversation Storage
# ======================

conversation = []

# ======================
# Helper Functions
# ======================

def format_conversation():
    if not conversation:
        return "No discussion yet."
    return "\n".join([f"{speaker}: {text}" for speaker, text in conversation])


def call_agent(agent_name, phase_instruction):
    system_prompt = AGENTS[agent_name]["identity"]

    user_prompt = f"""
Meeting topic:
{TOPIC}

Discussion so far:
{format_conversation()}

{phase_instruction}

Now speak as {agent_name}.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=300
    )

    content = response.choices[0].message.content

    if content is None:
        return "[No response]"

    return content.strip()


# ======================
# Meeting Phases
# ======================

PHASES = [
    "State your initial position on this topic.",
    "Critically respond to at least one previous viewpoint.",
    "Move toward a concrete implementation proposal.",
    "State your final recommendation clearly."
]

# ======================
# Run Meeting
# ======================
display(Markdown(f"## Meeting Topic: {TOPIC}"))

for i, phase in enumerate(PHASES):
    # print(f"\n===== Round {i+1} =====\n")
    display(Markdown(f"#### ===== Round {i+1} ====="))

    for agent in AGENTS.keys():
        reply = call_agent(agent, phase)
        conversation.append((agent, reply))

        display(Markdown(f"#### {agent} :"), Markdown(reply))

# ======================
# Final Summary by Architect
# ======================

# print("\n===== Final Architecture Decision =====\n")
display(Markdown(f"#### ===== Final Architecture Decision ====="))

final_summary_instruction = """
Summarize the discussion and propose the final architecture decision clearly.
"""

final_decision = call_agent("Architect", final_summary_instruction)
print(f"Architect (Final Decision): \n")
display(Markdown(final_decision))

## Meeting Topic: Should we implement a three-layer abstraction architecture for our multi-agent LLM system?

#### ===== Round 1 =====

#### Architect :

Three-layer abstraction (e.g., **Agents Layer** → **Service Layer** → **Infrastructure Layer**) is ideal for your multi-agent LLM system—it enforces separation of concerns, scalability, and future-proof modularity. Validate if domain-specific agent logic (e.g., task orchestration) can *clearly* decouple from runtime dependencies (e.g., LLM API calls, storage). If yes, proceed; if no, prefer a *tightly coupled yet explicit* dual-layer (e.g., **Domain Logic** + **Execution**). *Rule of thumb*: Assume complexity grows with agents—abstract early to avoid refactoring hell.

#### Engineer :

...

#### Researcher :

**Final recommendation:** Adopt a **boundary-hardened two-layer architecture (Orchestration + Execution)** but enforce *three ironclad constraints* **today** (via automated checks/CI):
1. **Dependency Inversion + Swappability** – All external dependencies (LLMs, APIs) must be *runtime-injectable* interfaces (`IFoo: Protocol`) with mandatory mocks; fail builds on hardcoded dependencies.
2. **Layer Isolation Enforcement** – Use static analysis (e.g., `pylint`/`eslint`) to block *any* shared state/mutable references between layers; enforce immutable task contexts passed via DI.
3. **Failure Mode Separation** – Require distinct exception types (`OrchestrationError`, `ExecutionFailure`) and log *layer-specific* failure cascades (flag >5% of failures crossing layers as red).

**Only refactor to three layers if**:
- >30% of changes touch both layers *and*
- Operational pain emerges (e.g., cascading failures, LLM swaps requiring orchestration changes).
*Metric*: If pain persists after 3 months, quantify the **cost of migration** (e.g., "30 engineer-days to split Execution into a separate layer")—then decide. **Key: Enforce boundaries as *non-negotiable gates*, not optional advice.**

#### ===== Final Architecture Decision =====

Architect (Final Decision): 



**Final Decision: Boundary-Hardened Two-Layer Architecture with Three Ironclad Constraints**
Start with a **two-layer scaffold** (Orchestration + Execution) but *mandate* **three executable boundaries** via automated enforcement:
1. **Dependency Inversion** – Enforce runtime-swappable interfaces (`IFoo`) with mock-first DI (block PRs without `MockLLM`/`MockDB`).
2. **Layer Isolation** – Use static analysis to block shared state/mutable references (e.g., `pylint`/`eslint` rules); enforce immutable task contexts passed via DI.
3. **Failure Mode Separation** – Require distinct exceptions (`OrchestrationError`/`ExecutionFailure`) and log *layer-specific* failure cascades (alert on >5% cross-layer failures).

**Refactor only if**:
- >30% of changes touch both layers *and*
- Operational pain surfaces (e.g., cascading failures, LLM swaps requiring orchestration edits).
*Default to "no refactor"* unless metrics/bottlenecks *prove* it’s required—**abstraction follows pain, not guesswork.** Deploy the constraints *today* as CI gates; iterate aggressively on metrics.